# 02 — Generate Motion History Images (MHI)

Converts every raw gait video into a single Motion History Image, then
builds the **subject-wise** train/validation/test split and materializes
`data/processed/{train,validation,test}/<class>/*.png`.

This notebook is a thin, inspectable wrapper around
`src/preprocessing/preprocessing.py` — run the full pipeline here cell by
cell so you can visually sanity-check the MHIs before training on them.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt
import pandas as pd

from src.preprocessing.mhi_generator import compute_mhi
from src.preprocessing.preprocessing import (
    build_metadata_table, generate_all_mhis, subject_wise_split,
    materialize_processed_dataset, run_full_preprocessing,
)
from src.utils.config import DEFAULT_CONFIG, METADATA_DIR
from src.utils.seed import set_global_seed

set_global_seed(DEFAULT_CONFIG.random_seed)
%matplotlib inline

## Step-by-step (recommended the first time you run this)

Each stage below mirrors one function in `preprocessing.py`. Running them
separately lets you inspect intermediate results (e.g. look at a couple of
MHIs) before committing to the full batch job. Once you trust the
pipeline, skip to the "One-shot" cell at the bottom.

### Step 1 — scan raw videos into a metadata table

In [ ]:
df = build_metadata_table()
print(f"{len(df)} videos, {df['subject_id'].nunique()} subjects")
df.head()

### Step 2 — preview the MHI computation on a single video before batch-processing everything

In [ ]:
cfg = DEFAULT_CONFIG
sample_row = df.iloc[0]
print("Sample video:", sample_row["video_path"], "| class:", sample_row["class"])

mhi = compute_mhi(
    sample_row["video_path"],
    tau=cfg.mhi_duration,
    threshold=cfg.binary_threshold,
    decay=cfg.mhi_decay,
    resize_to=cfg.image_size,
)

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(mhi, cmap="gray")
ax.set_title(f"MHI — {sample_row['class']} ({sample_row['severity']})")
ax.axis("off")
plt.show()

Brighter pixels = more recent motion; darker (but non-zero) pixels = older
motion within the `tau`-frame window; pure black = no motion detected there
at all. If the whole image looks empty/black, `binary_threshold` in
`src/utils/config.py` is probably too high for your footage — lower it and
re-run the cell above.

### Step 3 — batch-generate MHIs for every video (writes to `data/mhi_images/`)

In [ ]:
df = generate_all_mhis(df, cfg)
print(f"{len(df)} videos successfully converted to MHI.")
df.head()

### Step 4 — subject-wise train/validation/test split (leakage-free)

In [ ]:
df = subject_wise_split(df, cfg)
df["split"].value_counts()

In [ ]:
# Sanity check: no subject should appear in more than one split.
leak_check = df.groupby("subject_id")["split"].nunique()
n_leaking = (leak_check > 1).sum()
print(f"Subjects appearing in more than one split: {n_leaking} (should be 0)")
assert n_leaking == 0, "Data leakage detected -- a subject spans multiple splits!"


### Step 5 — materialize `data/processed/<split>/<class>/*.png` and write metadata CSVs

In [ ]:
materialize_processed_dataset(df)

os.makedirs(METADATA_DIR, exist_ok=True)
df.to_csv(os.path.join(METADATA_DIR, "labels.csv"), index=False)
df[["subject_id", "class"]].drop_duplicates().to_csv(os.path.join(METADATA_DIR, "subjects.csv"), index=False)
df.groupby(["class", "severity", "split"]).size().reset_index(name="count").to_csv(
    os.path.join(METADATA_DIR, "dataset_statistics.csv"), index=False
)
df[["subject_id", "class", "split"]].drop_duplicates().to_csv(
    os.path.join(METADATA_DIR, "train_val_test_split.csv"), index=False
)
print("Metadata written to", METADATA_DIR)

## Visual check: a few generated MHIs per class

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for row_idx, class_name in enumerate(cfg.class_names):
    samples = df[df["class"] == class_name].head(4)
    for col_idx in range(4):
        ax = axes[row_idx, col_idx]
        if col_idx < len(samples):
            img_path = samples.iloc[col_idx]["mhi_path"]
            img = plt.imread(img_path)
            ax.imshow(img, cmap="gray")
            ax.set_title(f"{class_name}", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

## One-shot alternative

In [ ]:
# Once you're confident in the pipeline, this single call does everything
# in Steps 1-5 above (and is exactly what `python -m src.preprocessing.preprocessing` runs):
#
# df = run_full_preprocessing(cfg)

## Next step

Move on to **`03_training.ipynb`** to train a backbone (VGG16 / MobileNet /
DenseNet) on the MHI images just generated in
`data/processed/{train,validation,test}/`.